In [ ]:
import pathlib, sys, base64
_root = pathlib.Path('..').resolve()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import polars as pl
from polars2svg import Polars2SVG
from IPython.display import HTML, display

p2s = Polars2SVG()
GOLDEN_DIR = pathlib.Path('../tests/golden')
GOLDEN_PNG_DIR = pathlib.Path('../tests/golden_png')


def compare(name, viz, df=None):
    golden_svg = (GOLDEN_DIR / f'{name}.svg').read_text()
    fresh_svg  = viz._repr_svg_()
    png_path = GOLDEN_PNG_DIR / f'{name}.png'
    png_row = ''
    if png_path.exists():
        b64 = base64.b64encode(png_path.read_bytes()).decode()
        png_row = f"""
      <tr>
        <th style="padding:4px 12px;font-size:12px;text-align:left;color:#555">PNG Golden &mdash; {name}</th>
        <th style="padding:4px 12px;font-size:12px;text-align:left;color:#555">Fresh SVG (browser-rendered)</th>
      </tr>
      <tr>
        <td style="padding:8px;vertical-align:top"><img src="data:image/png;base64,{b64}"/></td>
        <td style="padding:8px;vertical-align:top">{fresh_svg}</td>
      </tr>"""
    html = f"""
    <table style="border-collapse:collapse">
      <tr>
        <th style="padding:4px 12px;font-size:12px;text-align:left">Golden SVG &mdash; {name}</th>
        <th style="padding:4px 12px;font-size:12px;text-align:left">Fresh SVG Render</th>
      </tr>
      <tr>
        <td style="padding:8px;vertical-align:top">{golden_svg}</td>
        <td style="padding:8px;vertical-align:top">{fresh_svg}</td>
      </tr>{png_row}
    </table>"""
    if df is not None:
        display(df)
    display(HTML(html))


---
## XYp — Scatter / Lines / Distributions

### basic_scatter

In [ ]:
df = pl.DataFrame({
    'x': [1, 2, 3, 4, 5, 6],
    'y': [3, 1, 4, 1, 5, 9],
})
viz = p2s.xyp(df, 'x', 'y', wxh=(200, 200))
compare('basic_scatter', viz, df)

### color_continuous

In [ ]:
df = pl.DataFrame({
    'x':     [1, 2, 3, 4, 5, 6, 7, 8],
    'y':     [4, 2, 7, 1, 6, 3, 8, 5],
    'value': [10, 20, 30, 40, 50, 60, 70, 80],
})
viz = p2s.xyp(df, 'x', 'y', color='value', dot_size=4, wxh=(200, 200))
compare('color_continuous', viz, df)

### color_categorical

In [ ]:
df = pl.DataFrame({
    'x':     [1, 2, 3, 4, 5, 6],
    'y':     [3, 1, 4, 1, 5, 9],
    'group': ['a', 'b', 'a', 'b', 'a', 'b'],
})
viz = p2s.xyp(df, 'x', 'y', color='group', dot_size=4, wxh=(200, 200))
compare('color_categorical', viz, df)

### variable_dot_size

In [ ]:
df = pl.DataFrame({
    'x':    [1, 2, 3, 4, 5],
    'y':    [5, 3, 4, 2, 6],
    'size': [1, 3, 2, 5, 4],
})
viz = p2s.xyp(df, 'x', 'y', dot_size='size', wxh=(200, 200))
compare('variable_dot_size', viz, df)

### distributions

In [ ]:
df = pl.DataFrame({
    'x': [1, 1, 2, 2, 3, 3, 3, 4, 5, 5],
    'y': [2, 4, 1, 3, 2, 3, 5, 4, 1, 3],
})
viz = p2s.xyp(df, 'x', 'y',
              x_distributions=p2s.ROW_COUNTp,
              y_distributions=p2s.ROW_COUNTp,
              wxh=(200, 200))
compare('distributions', viz, df)

### lines

In [ ]:
df = pl.DataFrame({
    'time':   [0, 1, 2, 3, 4, 5, 6, 7],
    'value':  [2, 3, 1, 4, 2, 5, 3, 4],
    'series': ['a', 'a', 'a', 'a', 'b', 'b', 'b', 'b'],
})
viz = p2s.xyp(df, 'time', 'value', color='series', line=('series',), wxh=(200, 200))
compare('lines', viz, df)

### no_context

In [ ]:
df = pl.DataFrame({
    'x': [1, 2, 3, 4, 5],
    'y': [5, 1, 4, 2, 3],
})
viz = p2s.xyp(df, 'x', 'y', draw_context=False, wxh=(100, 100))
compare('no_context', viz, df)

---
## LinkP — Network / Relationship Diagrams

In [ ]:
# Shared data for all LinkP images
_DF_ = pl.DataFrame({
    'fm':       ['a',     'b',     'c',     'd',     'b'],
    'to':       ['b',     'c',     'd',     'a',     'a'],
    'category': ['cat_x', 'cat_y', 'cat_y', 'cat_x', 'cat_x'],
    'cat_n':    [10,      12,      12,      10,      10],
    'count':    [2.0,     5.0,     10.0,    0.1,     0.5],
})
_POS_ = {'a': (0.0, 0.5), 'b': (0.5, 0.0), 'c': (1.0, 0.5), 'd': (0.5, 1.0)}
_BASE_ = dict(df=_DF_, relationships=[('fm', 'to')], pos=_POS_,
              wxh=(96, 96), link_shape='curve', draw_labels=True, insets=(16, 16))
display(_DF_)

### linkp_node_color_none

In [ ]:
viz = p2s.linkp(**_BASE_, node_color=None)
compare('linkp_node_color_none', viz)

### linkp_node_color_hex
All nodes colored with a single hex value.

In [ ]:
viz = p2s.linkp(**_BASE_, node_color='#ff0000')
compare('linkp_node_color_hex', viz)

### linkp_node_color_dict_single
Only node `a` is colored; others use the default.

In [ ]:
viz = p2s.linkp(**_BASE_, node_color={'a': '#ff0000'})
compare('linkp_node_color_dict_single', viz)

### linkp_node_color_dict_two
Two nodes colored.

In [ ]:
viz = p2s.linkp(**_BASE_, node_color={'b': '#ff0000', 'd': '#00ff00'})
compare('linkp_node_color_dict_two', viz)

### linkp_node_color_dict_three
Three nodes colored (note: `'#999'` shorthand).

In [ ]:
viz = p2s.linkp(**_BASE_, node_color={'c': '#ff0000', 'd': '#999'})
compare('linkp_node_color_dict_three', viz)

### linkp_node_color_dict_unknown_key
Dict contains key `z` which is not a node — should be silently ignored.

In [ ]:
viz = p2s.linkp(**_BASE_, node_color={'c': '#ff0000', 'd': '#999', 'z': '#fff'})
compare('linkp_node_color_dict_unknown_key', viz)

---
## Smallp — Small Multiples

In [ ]:
# Shared data for all Smallp images: 5 categories (a–e)
df0 = pl.DataFrame({'a': [1,2,3],         'b': [4,5,6],         'cat': ['a','a','a']})
df1 = pl.DataFrame({'a': [3,4,5],         'b': [7,9,8],         'cat': ['b','b','b']})
df2 = pl.DataFrame({'a': [6,7,8],         'b': [1,3,5],         'cat': ['c','c','c']})
df3 = pl.DataFrame({'a': [2,3,4],         'b': [1,3,4],         'cat': ['d','d','d']})
df4 = pl.DataFrame({'a': [1,2,3,4,5,6,7,8], 'b': [8,8,7,7,6,6,5,5],
                    'cat': ['e','e','e','e','e','e','e','e']})
_SDF_ = pl.concat([df0, df1, df2, df3, df4])
_XYTP_ = p2s.xyp(df=_SDF_, x='a', y='b', color='cat',
                  line=('cat', 1.5), dot_size=2.0, wxh=(128, 128))
display(_SDF_)

### smallp_include_all_fits_exactly
`include_all=True`, `wxh=(384,384)` → 3×2 grid = 6 slots for 5 cats + `__all__` — fits exactly, no remainder.

In [ ]:
viz = p2s.smallp(_SDF_, 'cat', _XYTP_, include_all=True, wxh=(384, 384))
compare('smallp_include_all_fits_exactly', viz)

### smallp_basic_all_fit
No `include_all`, `wxh=(384,384)` → 6 slots for 5 categories — all fit, no remainder.

In [ ]:
viz = p2s.smallp(_SDF_, 'cat', _XYTP_, wxh=(384, 384))
compare('smallp_basic_all_fit', viz)

### smallp_small_wxh_single_slot_all_remainder
`wxh=(190,190)` → 1 slot for 5 categories — all 5 collated into `__remainder__`.

In [ ]:
viz = p2s.smallp(_SDF_, 'cat', _XYTP_, wxh=(190, 190))
compare('smallp_small_wxh_single_slot_all_remainder', viz)

### smallp_2x1_grid_with_remainder
`wxh=(256,190)` → 2×1 grid = 2 slots: top category + `__remainder__`.

In [ ]:
viz = p2s.smallp(_SDF_, 'cat', _XYTP_, wxh=(256, 190))
compare('smallp_2x1_grid_with_remainder', viz)

### smallp_2x2_grid_with_remainder
`wxh=(256, 2*143)` → 2×2 grid = 4 slots: 3 categories + `__remainder__`.

In [ ]:
viz = p2s.smallp(_SDF_, 'cat', _XYTP_, wxh=(256, 2*143))
compare('smallp_2x2_grid_with_remainder', viz)

### smallp_2x2_grid_include_all_with_remainder
`wxh=(256, 2*143)`, `include_all=True` → 4 slots: `__all__` + 2 categories + `__remainder__`.

In [ ]:
viz = p2s.smallp(_SDF_, 'cat', _XYTP_, wxh=(256, 2*143), include_all=True)
compare('smallp_2x2_grid_include_all_with_remainder', viz)

### smallp_2x2_grid_include_all_no_collate_remainder
`include_all=True`, `collate_remainder=False` → overflow categories are silently dropped rather than collated.

In [ ]:
viz = p2s.smallp(_SDF_, 'cat', _XYTP_, wxh=(256, 2*143),
                 include_all=True, collate_remainder=False)
compare('smallp_2x2_grid_include_all_no_collate_remainder', viz)

---
## ChordP — Chord Diagrams

In [ ]:
# Shared data for all ChordP images
_CDF_ = pl.DataFrame({
    'fm':       ['a', 'b', 'c', 'd', 'a', 'c'],
    'to':       ['b', 'c', 'd', 'a', 'c', 'a'],
    'category': ['x', 'y', 'y', 'x', 'x', 'y'],
    'weight':   [2.0, 5.0, 3.0, 1.0, 4.0, 2.0],
})
_CBASE_ = dict(df=_CDF_, relationships=[('fm', 'to')], wxh=(128, 128))
display(_CDF_)

### chordp_default

In [ ]:
viz = p2s.chordp(**_CBASE_)
compare('chordp_default', viz)

### chordp_node_color_hex
All nodes colored with a single hex value.

In [ ]:
viz = p2s.chordp(**_CBASE_, node_color='#ff0000')
compare('chordp_node_color_hex', viz)

### chordp_node_color_by_name
Node color driven by node name hash (COLOR_BY_NODE_NAME).

In [ ]:
viz = p2s.chordp(**_CBASE_, node_color=p2s.COLOR_BY_NODE_NAME)
compare('chordp_node_color_by_name', viz)

### chordp_link_color_src
Each link takes the color of its source node.

In [ ]:
viz = p2s.chordp(**_CBASE_, color='src')
compare('chordp_link_color_src', viz)

### chordp_link_color_vary
Link color driven by `category` field.

In [ ]:
viz = p2s.chordp(**_CBASE_, color='category')
compare('chordp_link_color_vary', viz)

### chordp_link_shape_bundled
Edges routed through a geometric skeleton (bundled mode).

In [ ]:
viz = p2s.chordp(**_CBASE_, link_shape='bundled')
compare('chordp_link_shape_bundled', viz)

### chordp_draw_labels_radial
Radial labels at arc midpoints (default label style).

In [ ]:
viz = p2s.chordp(**_CBASE_, draw_labels=True)
compare('chordp_draw_labels_radial', viz)

### chordp_node_selection
Selected nodes rendered on outer ring, others inset.

In [ ]:
viz = p2s.chordp(**_CBASE_, node_selection={'a', 'c'})
compare('chordp_node_selection', viz)

### chordp_count_weight
Arc and link sizes proportional to `weight` field sum.

In [ ]:
viz = p2s.chordp(**_CBASE_, count='weight')
compare('chordp_count_weight', viz)

### chordp_node_size_vary
Arc height proportional to node count.

In [ ]:
viz = p2s.chordp(**_CBASE_, node_size='vary')
compare('chordp_node_size_vary', viz)

---
## SpreadLinesP — Egocentric Temporal Influence

In [ ]:
_SLDF_ = pl.DataFrame({
    'fm':   ['a', 'b', 'c', 'a', 'd', 'b', 'c', 'a'],
    'to':   ['b', 'a', 'a', 'c', 'a', 'c', 'b', 'd'],
    'time': [1,   1,   1,   2,   2,   2,   3,   3  ],
    'role': ['x', 'y', 'y', 'x', 'y', 'y', 'x', 'x'],
    'w':    [3,   1,   2,   4,   1,   2,   3,   1  ],
})
_SLRELS_ = [('fm', 'to')]
display(_SLDF_)

### spreadlinesp_basic_ego_a

In [ ]:
viz = p2s.spreadlinesp(_SLDF_, _SLRELS_, ego='a', time='time', wxh=(700, 300))
compare('spreadlinesp_basic_ego_a', viz)

### spreadlinesp_node_color_field
Node color driven by `role` field.

In [ ]:
viz = p2s.spreadlinesp(_SLDF_, _SLRELS_, ego='a', time='time',
                       node_color='role', wxh=(700, 300))
compare('spreadlinesp_node_color_field', viz)

### spreadlinesp_count_field
Circle size scaled by `w` field sum.

In [ ]:
viz = p2s.spreadlinesp(_SLDF_, _SLRELS_, ego='a', time='time',
                       count='w', wxh=(700, 300))
compare('spreadlinesp_count_field', viz)

### spreadlinesp_highlight_nodes
Nodes `b` and `c` visually highlighted.

In [ ]:
viz = p2s.spreadlinesp(_SLDF_, _SLRELS_, ego='a', time='time',
                       highlight_nodes={'b', 'c'}, wxh=(700, 300))
compare('spreadlinesp_highlight_nodes', viz)

---
## Legends & Colorbars — `legend=` (should-do #5)
_Source: `test_legend_golden.py`_

`legend=False` is the default (existing goldens untouched). `True` ≡ `'right'`;
positions `'right' | 'left' | 'top' | 'bottom'`; dict form adds `title` / `fmt` /
`max_items` / `order`. The strip is reserved **from** `wxh` — the plot shrinks,
the physical output size does not.

In [ ]:
_LXY_ = pl.DataFrame({
    'x':   [1, 2, 3, 4, 5, 6, 7, 8] * 4,
    'y':   [2, 4, 1, 8, 5, 7, 3, 6] * 4,
    'cat': ['a', 'b', 'c', 'a', 'b', 'c', 'd', 'e'] * 4,
    'val': [1.0, 2.5, 3.2, 0.5, 4.4, 2.2, 1.1, 9.9] * 4,
})
_LBAR_ = pl.DataFrame({
    'bin': ['alpha'] * 8 + ['beta'] * 6 + ['gamma'] * 4 + ['delta'] * 2,
    'grp': (['x', 'x', 'y', 'z'] * 5),
    'val': ([5.0, 2.0, 8.0, 1.0] * 5),
})

### legend_xyp_categorical_right
Categorical swatch list (CSETp auto-selected from a string color field).

In [ ]:
viz = p2s.xyp(_LXY_, 'x', 'y', color='cat', legend='right', wxh=(256, 192))
compare('legend_xyp_categorical_right', viz)

### legend_xyp_colorbar_left
Colorbar (magnitude spectrum auto-selected from a numeric color field).

In [ ]:
viz = p2s.xyp(_LXY_, 'x', 'y', color='val', legend='left', wxh=(256, 192))
compare('legend_xyp_colorbar_left', viz)

### legend_xyp_colorbar_bottom
Horizontal colorbar for `CROW_MAGNITUDEp` (title defaults to `rows`).

In [ ]:
viz = p2s.xyp(_LXY_, 'x', 'y', color=p2s.CROW_MAGNITUDEp, legend='bottom', wxh=(256, 192))
compare('legend_xyp_colorbar_bottom', viz)

### legend_xyp_dict_top
Dict spec: `pos='top'`, custom `title`, `max_items=3` with `... N more` overflow.

In [ ]:
viz = p2s.xyp(_LXY_, 'x', 'y', color='cat',
              legend={'pos': 'top', 'title': 'Category', 'max_items': 3}, wxh=(256, 192))
compare('legend_xyp_dict_top', viz)

### legend_histop_categorical_bottom
Stacked-bar segment colors with a horizontal categorical legend.

In [ ]:
viz = p2s.histop(_LBAR_, 'bin', color='grp', legend='bottom', wxh=(192, 256))
compare('legend_histop_categorical_bottom', viz)

### legend_histop_colorbar_right
Whole-bar spectrum coloring with a vertical colorbar.

In [ ]:
viz = p2s.histop(_LBAR_, 'bin', color='val', legend='right', wxh=(256, 256))
compare('legend_histop_colorbar_right', viz)